# 06 — Faster R-CNN: Controlled Hyperparameter Tuning

This notebook tunes the **two-stage anchor-based Faster R-CNN ResNet-50 FPN** detector using the **same fold 0, frame step 6, batch size 2, and six-epoch training budget** as the original baseline.

**Design:** keep the completed 3-fold baseline unchanged. Run independent experiments on fold 0 with fixed seed, evaluate with COCO AP, and save each run to its own Drive directory. The original validation split (2,846 images) is not used for tuning.

**Important:** these are **hyperparameter** experiments, not a custom-loss experiment. Changing an optimizer or image resolution does **not** count as modifying the actual loss function. Loss modification should be implemented and documented separately if required by the assignment.

The baseline fold-0 COCO mAP50–95 was approximately **0.1541** (from the earlier notebook). Reconfirm the exact baseline number from Drive before interpreting improvements.

## 1 — Environment and Drive
Use a Colab **GPU runtime**. No Kaggle API and no `kaggle.json` are needed; the dataset ZIP already lives on Drive. PyTorch/torchvision are left at their preinstalled, mutually compatible versions.

In [1]:
%pip -q install pycocotools

import gc, json, random, shutil, time, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from google.colab import drive

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
DEVICE = torch.device('cuda')
SEED, FOLD, EPOCHS, BATCH_SIZE = 42, 0, 6, 2
NUM_WORKERS = 0  # prevent DataLoader child-process teardown warnings

ROOT = Path('/content/drive/MyDrive/UCU_AI_HW2')
ARCHIVE_PATH = ROOT / 'visdrone_competition.zip'
SPLIT_PATH = ROOT / 'cv_splits_step6.csv'
BASELINE_DIR = ROOT / 'faster_rcnn' / 'fold_0'
TUNING_ROOT = ROOT / 'faster_rcnn_tuning'
TUNING_ROOT.mkdir(parents=True, exist_ok=True)
assert ARCHIVE_PATH.is_file(), ARCHIVE_PATH
assert SPLIT_PATH.is_file(), SPLIT_PATH

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Tuning outputs:', TUNING_ROOT)

Mounted at /content/drive
PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
GPU: Tesla T4
Tuning outputs: /content/drive/MyDrive/UCU_AI_HW2/faster_rcnn_tuning


## 2 — Extract dataset only if not already present
This extracts a fresh **local** copy for training and keeps the ZIP on Google Drive. A new Colab runtime loses `/content/visdrone`, so extraction is needed again; downloading from Kaggle is not.

In [2]:
LOCAL_DIR = Path('/content/visdrone')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset_root(base):
    for annotation_file in base.rglob('train/annotations_coco.json'):
        candidate = annotation_file.parent.parent
        if ((candidate / 'train/images').is_dir()
                and (candidate / 'val/annotations_coco.json').is_file()):
            return candidate
    return None

DATA_ROOT = find_dataset_root(LOCAL_DIR)
if DATA_ROOT is None:
    assert zipfile.is_zipfile(ARCHIVE_PATH), 'Dataset ZIP is invalid.'
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        members = archive.infolist()
        required = sum(item.file_size for item in members)
        free = shutil.disk_usage('/content').free
        print(f'Extraction needs {required/1e9:.2f} GB; available {free/1e9:.2f} GB')
        assert free > required * 1.10, 'Insufficient local disk space.'
        base = LOCAL_DIR.resolve()
        for item in members:
            dest = (LOCAL_DIR / item.filename).resolve()
            assert dest == base or base in dest.parents, f'Unsafe ZIP entry: {item.filename}'
        archive.extractall(LOCAL_DIR)
    DATA_ROOT = find_dataset_root(LOCAL_DIR)

assert DATA_ROOT is not None, 'Could not locate unpacked dataset.'
with (DATA_ROOT / 'train/annotations_coco.json').open() as f:
    train_coco = json.load(f)
assert len(train_coco['images']) == 24201
print('Dataset ready:', DATA_ROOT)
print('Train images:', len(train_coco['images']))

Extraction needs 10.98 GB; available 120.80 GB
Dataset ready: /content/visdrone
Train images: 24201


## 3 — Use **exact** pre-existing video-level folds
Never regenerate the split for tuning. Check image IDs, disjoint video sequences, and fold sizes.

In [3]:
sampled = pd.read_csv(SPLIT_PATH)
expected_cols = {'image_id', 'fold', 'clip'}
assert expected_cols.issubset(sampled.columns), expected_cols - set(sampled.columns)
assert len(sampled) == 4062 and sampled['image_id'].is_unique
assert set(sampled['fold'].astype(int)) == {0, 1, 2}
assert sorted(sampled['fold'].value_counts().tolist()) == [1349, 1356, 1357]

available_ids = {int(img['id']) for img in train_coco['images']}
assert set(sampled['image_id'].astype(int)).issubset(available_ids)
for fold in range(3):
    train_part = sampled.loc[sampled['fold'] != fold]
    val_part = sampled.loc[sampled['fold'] == fold]
    assert set(train_part['clip']).isdisjoint(set(val_part['clip'])), 'Video leakage!'
    print(f"Fold {fold}: train={len(train_part)} / val={len(val_part)}, "
          f"train clips={train_part['clip'].nunique()}, val clips={val_part['clip'].nunique()}")

train_ids = set(sampled.loc[sampled['fold'] != FOLD, 'image_id'].astype(int))
val_ids = set(sampled.loc[sampled['fold'] == FOLD, 'image_id'].astype(int))
assert len(train_ids) == 2705 and len(val_ids) == 1357
print('Tuning fold 0 ready.')

Fold 0: train=2705 / val=1357, train clips=37, val clips=19
Fold 1: train=2706 / val=1356, train clips=37, val clips=19
Fold 2: train=2713 / val=1349, train clips=38, val clips=18
Tuning fold 0 ready.


## 4 — Dataset and DataLoaders
COCO bbox `[x,y,w,h]` is converted to `xyxy`. Faster R-CNN uses labels **1..8**, with **0 as background**. Training `iscrowd=1` regions are omitted from targets, matching the existing baseline limitation; original crowd annotations are preserved for COCO validation.

In [4]:
image_info = {int(im['id']): im for im in train_coco['images']}
annotations_by_image = defaultdict(list)
for ann in train_coco['annotations']:
    annotations_by_image[int(ann['image_id'])].append(ann)

def image_path(info):
    rel = Path(info['file_name'])
    candidates = [
        DATA_ROOT / 'train' / rel,
        DATA_ROOT / 'train/images' / rel,
        DATA_ROOT / 'train/images' / rel.parent.name / rel.name,
        DATA_ROOT / rel,
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Missing image: {info['file_name']}")

class VisDroneDetectionDataset(Dataset):
    def __init__(self, image_ids):
        self.ids = sorted(int(x) for x in image_ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        image_id = self.ids[idx]
        with Image.open(image_path(image_info[image_id])) as im:
            image = im.convert('RGB')
            width, height = image.size
            image_tensor = pil_to_tensor(image).float().div_(255.0)
        boxes, labels, areas = [], [], []
        for ann in annotations_by_image[image_id]:
            if int(ann.get('iscrowd', 0)) == 1:
                continue
            cls = int(ann['category_id'])
            if not 1 <= cls <= 8:
                continue
            x, y, w, h = map(float, ann['bbox'])
            x1, y1 = max(0., min(x, width)), max(0., min(y, height))
            x2, y2 = max(0., min(x+w, width)), max(0., min(y+h, height))
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(cls)
            areas.append((x2-x1)*(y2-y1))
        return image_tensor, {
            'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1,4),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'area': torch.tensor(areas, dtype=torch.float32),
            'image_id': torch.tensor(image_id, dtype=torch.int64),
            'iscrowd': torch.zeros(len(labels), dtype=torch.int64),
        }

def collate_fn(batch):
    return tuple(zip(*batch))

def make_loaders():
    train_ds = VisDroneDetectionDataset(train_ids)
    val_ds = VisDroneDetectionDataset(val_ids)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
    return train_dl, val_dl

train_dl_check, val_dl_check = make_loaders()
images_check, targets_check = next(iter(train_dl_check))
assert len(images_check) == 2
assert all(torch.isfinite(t['boxes']).all().item() for t in targets_check)
assert all(((t['labels'] >= 1) & (t['labels'] <= 8)).all().item() for t in targets_check)
print('Data smoke test passed:', tuple(images_check[0].shape))
del train_dl_check, val_dl_check, images_check, targets_check

Data smoke test passed: (3, 765, 1360)


## 5 — Experiment grid
The baseline remains untouched. Each experiment changes one **design dimension**: optimizer recipe (optimizer and its appropriate learning rate), learning rate alone, or resolution alone. The first is **not** a pure optimizer-only causal comparison because its learning rate also changes. All three run on exactly the same fold, seed, batch size, and epoch count.

Set `RUN_EXPERIMENTS` to a subset to fit your GPU allowance. The two first experiments are lower-risk; `resolution_768` may use more GPU memory.

In [5]:
EXPERIMENTS = {
    'adamw_2e4': {
        'optimizer': 'AdamW', 'lr': 0.0002,
        'min_size': 640, 'max_size': 1024, 'rpn_post_train': 1000,
    },
    'sgd_1e3': {
        'optimizer': 'SGD', 'lr': 0.001,
        'min_size': 640, 'max_size': 1024, 'rpn_post_train': 1000,
    },
    'resolution_768': {
        'optimizer': 'SGD', 'lr': 0.002,
        'min_size': 768, 'max_size': 1280, 'rpn_post_train': 1000,
    },
}

# Begin with the first two if GPU time or memory is limited.
RUN_EXPERIMENTS = ['adamw_2e4', 'sgd_1e3', 'resolution_768']
assert all(name in EXPERIMENTS for name in RUN_EXPERIMENTS)

baseline_metrics_path = BASELINE_DIR / 'validation_metrics.csv'
if baseline_metrics_path.is_file():
    baseline_metrics = pd.read_csv(baseline_metrics_path)
    display(baseline_metrics)
else:
    print('Baseline metrics CSV not found; tuning can proceed, but comparison needs that CSV.')

config_path = TUNING_ROOT / 'experiment_grid.json'
with config_path.open('w') as f:
    json.dump({'fold': FOLD, 'epochs': EPOCHS, 'batch': BATCH_SIZE,
               'seed': SEED, 'experiments': EXPERIMENTS}, f, indent=2)
print('Experiments:', RUN_EXPERIMENTS)

,model,fold,epochs_completed,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_frame
0,Faster R-CNN R50-FPN,0,6,0.154145,0.303931,0.139557,0.048235,0.124787,0.34181,142.035246


Experiments: ['adamw_2e4', 'sgd_1e3', 'resolution_768']


## 6 — Model, optimizer, and COCO evaluation
Use exactly the **baseline Faster R-CNN architecture** and change only parameters specified by each experiment. For training and evaluation, model category IDs remain **1..8**. COCO AP uses original crowd annotations with `maxDets=[1,10,100]`.

In [6]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_model(cfg):
    model = fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
        min_size=cfg['min_size'],
        max_size=cfg['max_size'],
        box_detections_per_img=300,
        rpn_pre_nms_top_n_train=2000,
        rpn_post_nms_top_n_train=cfg['rpn_post_train'],
        rpn_pre_nms_top_n_test=1000,
        rpn_post_nms_top_n_test=500,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 9)
    return model.to(DEVICE)


def build_optimizer(model, cfg):
    if cfg['optimizer'] == 'SGD':
        return torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9,
                               weight_decay=0.0005)
    if cfg['optimizer'] == 'AdamW':
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'],
                                 weight_decay=0.0005)
    raise ValueError(cfg['optimizer'])


def coco_evaluate(model, val_loader, image_ids, name):
    model.eval()
    gt = {
        'images': [im for im in train_coco['images'] if int(im['id']) in image_ids],
        'annotations': [ann for ann in train_coco['annotations']
                        if int(ann['image_id']) in image_ids and 1 <= int(ann['category_id']) <= 8],
        'categories': [c for c in train_coco['categories'] if 1 <= int(c['id']) <= 8],
        'info': train_coco.get('info', {}),
        'licenses': train_coco.get('licenses', []),
    }
    coco_gt = COCO()
    coco_gt.dataset = gt
    coco_gt.createIndex()
    preds, timings = [], []
    with torch.inference_mode():
        for images, targets in tqdm(val_loader, desc=f'{name} validation', leave=False):
            inputs = [im.to(DEVICE) for im in images]
            torch.cuda.synchronize()
            started = time.perf_counter()
            outputs = model(inputs)
            torch.cuda.synchronize()
            timings.append((time.perf_counter()-started)/len(images))
            for output, target in zip(outputs, targets):
                image_id = int(target['image_id'].item())
                for bbox, confidence, label in zip(output['boxes'].cpu().tolist(),
                                                    output['scores'].cpu().tolist(),
                                                    output['labels'].cpu().tolist()):
                    if float(confidence) < 0.001 or not 1 <= int(label) <= 8:
                        continue
                    x1,y1,x2,y2 = bbox
                    if x2 <= x1 or y2 <= y1:
                        continue
                    preds.append({
                        'image_id': image_id, 'category_id': int(label),
                        'bbox': [float(x1),float(y1),float(x2-x1),float(y2-y1)],
                        'score': float(confidence),
                    })
    assert preds, f'{name} produced no predictions; inspect training.'
    coco_dt = coco_gt.loadRes(preds)
    evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
    evaluator.params.imgIds = sorted(image_ids)
    evaluator.params.catIds = list(range(1,9))
    evaluator.params.maxDets = [1,10,100]
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    s = evaluator.stats
    return {
        'mAP50-95': float(s[0]), 'AP50': float(s[1]), 'AP75': float(s[2]),
        'AP_small': float(s[3]), 'AP_medium': float(s[4]), 'AP_large': float(s[5]),
        'inference_ms_per_frame': float(np.mean(timings)*1000),
    }

print('Model and evaluator functions ready.')

Model and evaluator functions ready.


## 7 — Preflight one-batch forward and backward
Tests the baseline-size training step before the full experiment grid. This is not an optimization step. If `resolution_768` runs out of memory later, do not quietly reduce batch size and compare it as if the protocols were identical.

In [7]:
set_seed()
smoke_cfg = EXPERIMENTS['sgd_1e3']
smoke = build_model(smoke_cfg)
smoke.train()
smoke_train, _ = make_loaders()
smoke_images, smoke_targets = next(iter(smoke_train))
smoke_images = [im.to(DEVICE) for im in smoke_images]
smoke_targets = [{k:v.to(DEVICE) for k,v in t.items()} for t in smoke_targets]
smoke.zero_grad(set_to_none=True)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    smoke_losses = smoke(smoke_images, smoke_targets)
    smoke_total = sum(smoke_losses.values())
assert bool(torch.isfinite(smoke_total).item())
scaler_test = torch.amp.GradScaler('cuda')
scaler_test.scale(smoke_total).backward()
print('Forward + backward passed. Loss:', round(smoke_total.item(), 4))
print('GPU peak memory (GB):', round(torch.cuda.max_memory_allocated()/1e9,2))
del smoke, smoke_train, smoke_images, smoke_targets, smoke_losses, smoke_total, scaler_test
gc.collect(); torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 163MB/s]


Forward + backward passed. Loss: 4.7327
GPU peak memory (GB): 1.66


## 8 — Tuning training loop (checkpoint and resume)
Each experiment has its **own** Drive folder and can resume from the last completed epoch. Do **not** write anything into `/faster_rcnn/fold_0/`. No training happens until the next cell invokes `run_experiment`.

In [8]:
def save_checkpoint_atomic(payload, target_path, name):
    # Save to local SSD first, then copy to a temporary Drive file.
    local = Path('/content') / f'frcnn_tune_{name}_latest.pt'
    partial = target_path.with_name('latest_uploading.pt')
    torch.save(payload, local)
    shutil.copy2(local, partial)
    partial.replace(target_path)


def run_experiment(name):
    assert name in EXPERIMENTS
    cfg = EXPERIMENTS[name]
    output_dir = TUNING_ROOT / name
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / 'latest.pt'
    history_path = output_dir / 'history.csv'
    metrics_path = output_dir / 'validation_metrics.csv'

    if metrics_path.is_file() and checkpoint_path.is_file():
        record = pd.read_csv(metrics_path)
        if len(record) == 1 and int(record.iloc[0]['epochs_completed']) == EPOCHS:
            print(f'{name}: already completed; skipping.')
            return record

    set_seed()
    gc.collect(); torch.cuda.empty_cache()
    train_loader, val_loader = make_loaders()
    model = build_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    history, start_epoch = [], 0

    if checkpoint_path.is_file():
        state = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        assert state['experiment'] == name and state['cfg'] == cfg
        assert state['fold'] == FOLD and state['epochs'] == EPOCHS
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        history = state['history']
        start_epoch = int(state['epoch'])+1
        print(f'{name}: resuming at epoch {start_epoch+1}.')
        del state
    else:
        print(f'{name}: starting from COCO pretrained weights.')

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        start = time.perf_counter()
        loss_total = 0.0
        progress = tqdm(train_loader, desc=f'{name} | epoch {epoch+1}/{EPOCHS}', leave=False)
        for images, targets in progress:
            images_gpu = [im.to(DEVICE) for im in images]
            targets_gpu = [{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                losses = model(images_gpu, targets_gpu)
                total = sum(losses.values())
            if not bool(torch.isfinite(total).item()):
                raise FloatingPointError(f'{name}: nonfinite loss at epoch {epoch+1}')
            scaler.scale(total).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_total += float(total.item())
            progress.set_postfix(loss=f'{total.item():.4f}')
        scheduler.step()
        record = {
            'experiment': name, 'epoch': epoch+1,
            'train_loss': loss_total / len(train_loader),
            'learning_rate_next_epoch': optimizer.param_groups[0]['lr'],
            'minutes': (time.perf_counter()-start)/60,
        }
        history.append(record)
        save_checkpoint_atomic({
            'epoch': epoch, 'experiment': name, 'cfg': cfg,
            'fold': FOLD, 'epochs': EPOCHS, 'seed': SEED,
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
            'history': history,
        }, checkpoint_path, name)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f"{name}: epoch {epoch+1}/{EPOCHS}, loss={record['train_loss']:.4f}, "
              f"time={record['minutes']:.1f} min; checkpoint saved")

    print(f'{name}: evaluating fold {FOLD}...')
    measures = coco_evaluate(model, val_loader, val_ids, name)
    result = {'experiment': name, 'fold': FOLD, 'epochs_completed': EPOCHS,
              'optimizer': cfg['optimizer'], 'lr': cfg['lr'],
              'min_size': cfg['min_size'], 'max_size': cfg['max_size'],
              'batch': BATCH_SIZE, **measures}
    pd.DataFrame([result]).to_csv(metrics_path, index=False)
    print(f"{name}: mAP={result['mAP50-95']:.4f}, AP_small={result['AP_small']:.4f}")
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame([result])

print('Training function ready. No experiment has started yet.')

Training function ready. No experiment has started yet.


## 9 — Run the selected tuning experiments
Runs sequentially in the same notebook. If Colab disconnects, rerun setup cells and this cell; completed runs are skipped and incomplete runs resume after their last finished epoch. GPU resources are not guaranteed overnight.

In [9]:
for experiment_name in RUN_EXPERIMENTS:
    result = run_experiment(experiment_name)
    display(result[['experiment', 'fold', 'mAP50-95', 'AP50', 'AP_small', 'inference_ms_per_frame']])
print('Requested tuning experiments completed.')

adamw_2e4: already completed; skipping.


,experiment,fold,mAP50-95,AP50,AP_small,inference_ms_per_frame
0,adamw_2e4,0,0.154744,0.304474,0.047127,75.982758


sgd_1e3: already completed; skipping.


,experiment,fold,mAP50-95,AP50,AP_small,inference_ms_per_frame
0,sgd_1e3,0,0.150845,0.300472,0.046846,76.516003


resolution_768: already completed; skipping.


,experiment,fold,mAP50-95,AP50,AP_small,inference_ms_per_frame
0,resolution_768,0,0.167708,0.331819,0.056121,118.99528


Requested tuning experiments completed.


## 10 — Comparison against the original Faster R-CNN baseline
Metrics shown below use the same validation fold and COCO evaluator; however, the original baseline used a score cutoff of 0.001 and these experiments do too. **Only compare fold 0 with fold 0**. Do not compare tuning fold 0 against 3-fold baseline mean as though they were paired evaluations.

In [10]:
rows = []
if baseline_metrics_path.is_file():
    base = pd.read_csv(baseline_metrics_path).iloc[0]
    rows.append({
        'experiment': 'baseline_sgd_2e3_640',
        'fold': 0,
        'mAP50-95': float(base['mAP50-95']),
        'AP50': float(base['AP50']),
        'AP_small': float(base['AP_small']),
        'inference_ms_per_frame': float(base['inference_ms_per_frame']),
    })

for name in EXPERIMENTS:
    file = TUNING_ROOT / name / 'validation_metrics.csv'
    if file.is_file():
        df = pd.read_csv(file).iloc[0]
        rows.append({
            'experiment': name, 'fold': int(df['fold']),
            'mAP50-95': float(df['mAP50-95']),
            'AP50': float(df['AP50']),
            'AP_small': float(df['AP_small']),
            'inference_ms_per_frame': float(df['inference_ms_per_frame']),
        })

assert rows, 'No baseline or tuning results found.'
comparison = pd.DataFrame(rows).sort_values('mAP50-95', ascending=False)
comparison['delta_mAP_vs_baseline'] = np.nan
if 'baseline_sgd_2e3_640' in comparison['experiment'].values:
    baseline_map = float(comparison.loc[
        comparison['experiment'] == 'baseline_sgd_2e3_640', 'mAP50-95'].iloc[0])
    comparison['delta_mAP_vs_baseline'] = comparison['mAP50-95'] - baseline_map
comparison.to_csv(TUNING_ROOT / 'fold0_comparison.csv', index=False)
display(comparison.round(4))
print('Comparison saved:', TUNING_ROOT / 'fold0_comparison.csv')

,experiment,fold,mAP50-95,AP50,AP_small,inference_ms_per_frame,delta_mAP_vs_baseline
3,resolution_768,0,0.1677,0.3318,0.0561,118.9953,0.0136
1,adamw_2e4,0,0.1547,0.3045,0.0471,75.9828,0.0006
0,baseline_sgd_2e3_640,0,0.1541,0.3039,0.0482,142.0352,0.0000
2,sgd_1e3,0,0.1508,0.3005,0.0468,76.5160,-0.0033


Comparison saved: /content/drive/MyDrive/UCU_AI_HW2/faster_rcnn_tuning/fold0_comparison.csv


## 11 — Continue the selected `resolution_768` experiment on folds 1 and 2

**Run only these new cells at the bottom of your *already initialized* Colab notebook.** The original experiments on fold 0 are preserved, and the saved `resolution_768` fold-0 checkpoint is reused **without retraining**.

- Selected model: Faster R-CNN ResNet50-FPN, **SGD 0.002, min size 768, max size 1280**, six epochs, batch 2, seed 42.
- Original splits: `cv_splits_step6.csv`, **GroupKFold by video**.
- New output folders: `MyDrive/UCU_AI_HW2/faster_rcnn_tuning/resolution_768_3fold/fold_1` and `fold_2`.
- Each epoch saves a resumable `latest.pt` directly to Google Drive. `validation_metrics.csv` is written only after complete training and COCO evaluation.

**Important:** Do not re-run the earlier `for experiment_name in RUN_EXPERIMENTS` cell. Run the continuation cells below instead. On a *new* Colab runtime, rerun setup/definitions (cells above) first; completed experiments should be skipped, but you do not need to re-run their training loop.

In [11]:
from pathlib import Path
import gc, time, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SELECTED_NAME = 'resolution_768'
SELECTED_CFG = EXPERIMENTS[SELECTED_NAME].copy()
assert SELECTED_CFG == {
    'optimizer': 'SGD', 'lr': 0.002,
    'min_size': 768, 'max_size': 1280, 'rpn_post_train': 1000,
}, 'Unexpected tuned config. Inspect EXPERIMENTS before continuing.'
assert EPOCHS == 6 and BATCH_SIZE == 2 and SEED == 42
assert len(sampled) == 4062 and sampled['image_id'].is_unique
assert torch.cuda.is_available(), 'Enable Colab GPU.'

TUNED_CV_ROOT = TUNING_ROOT / 'resolution_768_3fold'
TUNED_CV_ROOT.mkdir(parents=True, exist_ok=True)
FOLD0_DIR = TUNING_ROOT / SELECTED_NAME
FOLD0_METRICS = FOLD0_DIR / 'validation_metrics.csv'
FOLD0_CKPT = FOLD0_DIR / 'latest.pt'
assert FOLD0_METRICS.is_file() and FOLD0_CKPT.is_file(), (
    'Fold 0 tuned training is missing. Check the original resolution_768 folder on Drive.'
)
f0 = pd.read_csv(FOLD0_METRICS)
assert len(f0) == 1 and int(f0.iloc[0]['fold']) == 0
assert int(f0.iloc[0]['epochs_completed']) == 6
assert int(f0.iloc[0]['min_size']) == 768
print('Fold 0 already complete — will NOT retrain:', FOLD0_DIR)
print('Fold 0 tuned mAP:', float(f0.iloc[0]['mAP50-95']))
print('New results folder:', TUNED_CV_ROOT)

Fold 0 already complete — will NOT retrain: /content/drive/MyDrive/UCU_AI_HW2/faster_rcnn_tuning/resolution_768
Fold 0 tuned mAP: 0.1677080068495938
New results folder: /content/drive/MyDrive/UCU_AI_HW2/faster_rcnn_tuning/resolution_768_3fold


### 11.1 — Fold-aware training (Drive checkpoints; safe resume)

This uses the existing `build_model`, `build_optimizer`, `VisDroneDetectionDataset`, `coco_evaluate`, and `save_checkpoint_atomic` helpers from the original notebook, but always builds **fresh loaders for the correct fold** and saves into its own fold folder. It **never** rewrites the original fold-0 checkpoint.

In [12]:
def make_fold_loaders(fold):
    assert fold in (1, 2), 'Fold 0 is already finished and must not be rerun.'
    train_part = sampled.loc[sampled['fold'].astype(int) != fold]
    val_part = sampled.loc[sampled['fold'].astype(int) == fold]
    assert set(train_part['clip']).isdisjoint(set(val_part['clip'])), 'Video leakage!'
    expected_val = {1: 1356, 2: 1349}[fold]
    assert len(val_part) == expected_val
    assert len(train_part) + len(val_part) == 4062
    tr_ids = set(train_part['image_id'].astype(int))
    va_ids = set(val_part['image_id'].astype(int))
    tr_ds = VisDroneDetectionDataset(tr_ids)
    va_ds = VisDroneDetectionDataset(va_ids)
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
    return tr_dl, va_dl, va_ids


def run_tuned_cv_fold(fold):
    assert fold in (1, 2)
    cfg = SELECTED_CFG.copy()
    label = f'{SELECTED_NAME}_fold_{fold}'
    out = TUNED_CV_ROOT / f'fold_{fold}'
    out.mkdir(parents=True, exist_ok=True)
    ckpt_path = out / 'latest.pt'
    history_path = out / 'history.csv'
    metrics_path = out / 'validation_metrics.csv'

    if metrics_path.is_file() and ckpt_path.is_file():
        done = pd.read_csv(metrics_path)
        if len(done) == 1 and int(done.iloc[0]['epochs_completed']) == EPOCHS \
                and int(done.iloc[0]['fold']) == fold:
            print(f'Fold {fold} already completed; skipping training and evaluation.')
            return done

    set_seed(SEED)
    gc.collect()
    torch.cuda.empty_cache()
    train_loader, val_loader, validation_ids = make_fold_loaders(fold)
    print(f'Fold {fold}: train={len(train_loader.dataset)}, val={len(val_loader.dataset)}')
    model = build_model(cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    history = []
    start_epoch = 0

    if ckpt_path.is_file():
        state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        assert state['experiment'] == SELECTED_NAME and state['cfg'] == cfg
        assert int(state['fold']) == fold and int(state['epochs']) == EPOCHS
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        history = state['history']
        start_epoch = int(state['epoch']) + 1
        assert 0 <= start_epoch <= EPOCHS
        print(f'Fold {fold}: resuming from epoch {start_epoch+1} (completed {start_epoch}).')
        del state
    else:
        print(f'Fold {fold}: fresh COCO-pretrained initialization (same as fold 0).')

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        started = time.perf_counter()
        total_loss = 0.0
        progress = tqdm(train_loader, desc=f'Fold {fold} | epoch {epoch+1}/{EPOCHS}', leave=False)
        for images, targets in progress:
            images_gpu = [im.to(DEVICE) for im in images]
            targets_gpu = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                losses = model(images_gpu, targets_gpu)
                loss = sum(losses.values())
            if not bool(torch.isfinite(loss).item()):
                raise FloatingPointError(f'Fold {fold}, epoch {epoch+1}: nonfinite loss')
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.item())
            progress.set_postfix(loss=f'{loss.item():.4f}')
        scheduler.step()
        epoch_record = {
            'experiment': SELECTED_NAME,
            'fold': fold,
            'epoch': epoch+1,
            'train_loss': total_loss / len(train_loader),
            'learning_rate_next_epoch': optimizer.param_groups[0]['lr'],
            'minutes': (time.perf_counter() - started) / 60,
        }
        history.append(epoch_record)
        save_checkpoint_atomic({
            'epoch': epoch, 'experiment': SELECTED_NAME,
            'cfg': cfg, 'fold': fold, 'epochs': EPOCHS, 'seed': SEED,
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
            'history': history,
        }, ckpt_path, label)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f"Fold {fold}: epoch {epoch+1}/{EPOCHS}, "
              f"loss={epoch_record['train_loss']:.4f}, "
              f"{epoch_record['minutes']:.1f} min. Checkpoint saved to Drive.")

    print(f'Fold {fold}: COCO evaluation on its own validation split...')
    scores = coco_evaluate(model, val_loader, validation_ids, label)
    result = {
        'experiment': SELECTED_NAME, 'fold': fold, 'epochs_completed': EPOCHS,
        'optimizer': cfg['optimizer'], 'lr': cfg['lr'],
        'min_size': cfg['min_size'], 'max_size': cfg['max_size'],
        'batch': BATCH_SIZE, **scores,
    }
    pd.DataFrame([result]).to_csv(metrics_path, index=False)
    print(f"Fold {fold} COMPLETED: mAP50-95={result['mAP50-95']:.5f}, "
          f"AP_small={result['AP_small']:.5f}")
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame([result])

print('Fold-aware training function ready. Nothing has been trained yet.')


Fold-aware training function ready. Nothing has been trained yet.


### 11.2 — Run folds 1 and 2 **one at a time**

Run the following cells in sequence. If Colab disconnects, repeat setup and function-definition cells; rerunning a **completed** fold skips it, and an interrupted one resumes from its last saved epoch. In a new runtime, `/content/visdrone` must be re-extracted, but the Drive checkpoints remain.

In [13]:
result_fold1 = run_tuned_cv_fold(1)
display(result_fold1)

Fold 1: train=2706, val=1356
Fold 1: fresh COCO-pretrained initialization (same as fold 0).


Fold 1 | epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Fold 1: epoch 1/6, loss=1.1278, 11.7 min. Checkpoint saved to Drive.


Fold 1 | epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Fold 1: epoch 2/6, loss=0.9020, 11.6 min. Checkpoint saved to Drive.


Fold 1 | epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (6 outputs left out)

Fold 1: epoch 6/6, loss=0.6468, 11.6 min. Checkpoint saved to Drive.
Fold 1: COCO evaluation on its own validation split...
creating index...
index created!


resolution_768_fold_1 validation:   0%|          | 0/678 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.16s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=53.98s).
Accumulating evaluation results...
DONE (t=1.30s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.162
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.313
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.148
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.037
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.158
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.368
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.090
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.194
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.248
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,experiment,fold,epochs_completed,optimizer,lr,min_size,max_size,batch,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_frame
0,resolution_768,1,6,SGD,0.002,768,1280,2,0.162383,0.313317,0.148393,0.036555,0.15825,0.368263,116.943419


In [14]:
result_fold2 = run_tuned_cv_fold(2)
display(result_fold2)

Fold 2: train=2713, val=1349
Fold 2: fresh COCO-pretrained initialization (same as fold 0).


Fold 2 | epoch 1/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Fold 2: epoch 1/6, loss=1.0996, 11.8 min. Checkpoint saved to Drive.


Fold 2 | epoch 2/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Fold 2: epoch 2/6, loss=0.8805, 11.8 min. Checkpoint saved to Drive.


Fold 2 | epoch 3/6:   0%|          | 0/1357 [00:00<?, ?it/s]

... (6 outputs left out)

Fold 2: epoch 6/6, loss=0.6308, 11.9 min. Checkpoint saved to Drive.
Fold 2: COCO evaluation on its own validation split...
creating index...
index created!


resolution_768_fold_2 validation:   0%|          | 0/675 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=38.83s).
Accumulating evaluation results...
DONE (t=1.89s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.171
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.336
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.159
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.078
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.164
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.300
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.112
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.226
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.267
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

,experiment,fold,epochs_completed,optimizer,lr,min_size,max_size,batch,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_frame
0,resolution_768,2,6,SGD,0.002,768,1280,2,0.171254,0.33612,0.158661,0.078049,0.164104,0.299547,110.319576


## 12 — Final paired baseline vs tuned 3-fold comparison

Run this only after both new folds finish. It loads the **existing** fold-0 tuned metrics and baseline evaluation files for all folds, produces per-fold paired deltas, and saves summary CSVs to Google Drive. No training is performed here.

These scores are valid within the shared torchvision COCOeval protocol; they should **not** be compared numerically with Ultralytics YOLO mAP without re-evaluation under a common protocol.

In [15]:
tuned_metrics_files = {
    0: FOLD0_METRICS,
    1: TUNED_CV_ROOT / 'fold_1' / 'validation_metrics.csv',
    2: TUNED_CV_ROOT / 'fold_2' / 'validation_metrics.csv',
}
baseline_metrics_files = {
    fold: ROOT / 'faster_rcnn' / f'fold_{fold}' / 'validation_metrics.csv'
    for fold in (0, 1, 2)
}
for fold in (0, 1, 2):
    assert tuned_metrics_files[fold].is_file(), f'Tuned fold {fold} missing: {tuned_metrics_files[fold]}'
    assert baseline_metrics_files[fold].is_file(), f'Baseline fold {fold} missing: {baseline_metrics_files[fold]}'

records = []
metrics_cols = ['mAP50-95', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']
for fold in (0, 1, 2):
    for variant, path in [('baseline_640', baseline_metrics_files[fold]),
                          ('tuned_768', tuned_metrics_files[fold])]:
        frame = pd.read_csv(path)
        assert len(frame) == 1
        rec = frame.iloc[0]
        assert int(rec['fold']) == fold
        assert all(col in frame.columns for col in metrics_cols)
        records.append({'fold': fold, 'variant': variant,
                        **{col: float(rec[col]) for col in metrics_cols}})

cv_comparison = pd.DataFrame(records).sort_values(['fold', 'variant'])
paired = cv_comparison.pivot(index='fold', columns='variant', values=metrics_cols)
paired_delta = pd.DataFrame({'fold': [0, 1, 2]})
for col in metrics_cols:
    paired_delta[f'{col}_baseline'] = paired[(col, 'baseline_640')].to_numpy()
    paired_delta[f'{col}_tuned'] = paired[(col, 'tuned_768')].to_numpy()
    paired_delta[f'{col}_delta'] = (paired[(col, 'tuned_768')] - paired[(col, 'baseline_640')]).to_numpy()

summary_rows = []
for variant, group in cv_comparison.groupby('variant'):
    entry = {'variant': variant, 'n_folds': len(group)}
    for col in metrics_cols:
        entry[f'{col}_mean'] = float(group[col].mean())
        entry[f'{col}_std'] = float(group[col].std(ddof=1))
    summary_rows.append(entry)
cv_summary = pd.DataFrame(summary_rows).sort_values('variant')

cv_comparison.to_csv(TUNED_CV_ROOT / 'baseline_vs_tuned_per_fold.csv', index=False)
paired_delta.to_csv(TUNED_CV_ROOT / 'baseline_vs_tuned_paired_delta.csv', index=False)
cv_summary.to_csv(TUNED_CV_ROOT / 'baseline_vs_tuned_cv_summary.csv', index=False)

print('Paired fold deltas:')
display(paired_delta[['fold', 'mAP50-95_baseline', 'mAP50-95_tuned', 'mAP50-95_delta',
                      'AP_small_baseline', 'AP_small_tuned', 'AP_small_delta']].round(5))
print('Three-fold means and standard deviations:')
display(cv_summary[['variant', 'mAP50-95_mean', 'mAP50-95_std',
                    'AP50_mean', 'AP_small_mean']].round(5))
print('CSV files saved:', TUNED_CV_ROOT)


Paired fold deltas:


,fold,mAP50-95_baseline,mAP50-95_tuned,mAP50-95_delta,AP_small_baseline,AP_small_tuned,AP_small_delta
0,0,0.15414,0.16771,0.01356,0.04823,0.05612,0.00789
1,1,0.13896,0.16238,0.02342,0.03205,0.03655,0.00451
2,2,0.14358,0.17125,0.02767,0.09272,0.07805,-0.01467


Three-fold means and standard deviations:


,variant,mAP50-95_mean,mAP50-95_std,AP50_mean,AP_small_mean
0,baseline_640,0.14556,0.00778,0.28798,0.05767
1,tuned_768,0.16712,0.00446,0.32709,0.05691


CSV files saved: /content/drive/MyDrive/UCU_AI_HW2/faster_rcnn_tuning/resolution_768_3fold


### Interpretation reminder

Fold 0 was used to **select** resolution 768, so the 3-fold tuned score is a useful consistency check but not a fully independent nested-CV estimate. Compare paired folds and report the tuning protocol honestly. A separate original holdout or untouched test set can be used for a final evaluation if the assignment permits.


## Final Summary — Faster R-CNN Hyperparameter Tuning

### Experimental setup
- Architecture: Faster R-CNN ResNet-50 FPN
- Cross-validation: 3-fold GroupKFold by video
- Epochs: 6 per fold
- Batch size: 2
- Random seed: 42
- Evaluation: COCO mAP50–95 with `iscrowd` handling

### Hyperparameter experiments

Three configurations were tested against the original
baseline on fold 0:
1. AdamW, learning rate 0.0002
2. SGD, learning rate 0.001
3. SGD, learning rate 0.002, image size 768–1280

The resolution experiment achieved the highest fold-0
mAP50–95 and was selected for further evaluation.

### Three-fold cross-validation results

| Metric | Baseline | Tuned |
|---|---:|---:|
| Mean mAP50–95 | 0.14556 | 0.16712 |
| Fold 0 mAP | 0.15414 | 0.16771 |
| Fold 1 mAP | 0.13896 | 0.16238 |
| Fold 2 mAP | 0.14358 | 0.17125 |

The tuned configuration improved mean mAP50–95 by
0.02156, corresponding to approximately 14.8% relative
improvement.

### Selected configuration
- Optimizer: SGD
- Learning rate: 0.002
- Min image size: 768
- Max image size: 1280
- Batch size: 2
- Epochs: 6

### Observations
- The tuned model achieved higher overall mAP on all
  three validation folds.
- Mean AP_small did not improve consistently.
- Fold 0 was used for hyperparameter selection, so the
  final CV estimate is not fully independent of tuning.
- Baseline and tuned checkpoints were preserved
  separately on Google Drive.

### Status
Hyperparameter tuning and 3-fold evaluation completed.

A separate loss-modification experiment and the final
cross-architecture comparison remain future tasks.
